# Recurring Charge ML Experimentation

This notebook connects to the local PostgreSQL database populated from DynamoDB and implements the industry-standard multi-layered enrichment pipeline.

In [57]:
import psycopg2
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
import warnings
warnings.filterwarnings('ignore')

# Connect to PostgreSQL
conn = psycopg2.connect(
    host="db",
    port="5432",
    user="ml_user",
    password="ml_password",
    database="recurring_charges_ml"
)

# Load transactions
query = "SELECT * FROM transactions"
df_tx = pd.read_sql(query, conn)
print(f"Loaded {len(df_tx)} transactions.")

Loaded 3300 transactions.


## 1. Normalization & Cleaning Pipeline

Strip noise (IDs, dates), standardize entities (LTD, INC), and extract geography.

In [58]:
def clean_merchant_name(name):
    if not isinstance(name, str) or not name: return ""
    name = name.upper()
    
    # Remove Transaction IDs (*1234, #A1B2)
    name = re.sub(r'[*#]\s?(?=[A-Za-z]*\d)[A-Z0-9]+', ' ', name)
    
    # Remove Dates (24/02, etc)
    name = re.sub(r'\d{2}/\d{2}', ' ', name)
    
    # Remove Payment Methods & Region Codes (APPLEPAY, GB)
    name = re.sub(r'\b(?:APPLEPAY|GB|UK)\b', ' ', name)

    # 1. Remove Zettle and all its attached garbage (ZETTLE_, ZETTLE_*, IZETTLE)
    name = re.sub(r'\bI?ZETTLE[_*]*', ' ', name)
    
    # 2. Remove PayPal and its attached garbage (PAYPAL *, PAYPAL_)
    name = re.sub(r'\bPAYPAL\s*[*_]*', ' ', name)
    
    # 3. Standard Payment Methods & Region Codes
    name = re.sub(r'\b(?:APPLEPAY|SUMUP|SQUARE|GB|UK)\b', ' ', name)

    
    # Remove long standalone numbers (Store IDs, Card last 4, Account numbers)
    # Matches any standalone number that is 3 or more digits long (e.g., 2663, 9178, 1448432524)
    name = re.sub(r'\b\d{3,}\b', ' ', name)
    
    # Remove alphanumeric reference numbers (e.g., 08477317OB, T1024025862)
    # The (?=.*\d) ensures the 8+ character word actually contains a number
    # so we don't accidentally delete purely text names like "SAINSBURYS"
    name = re.sub(r'\b(?=[A-Za-z]*\d)[A-Za-z0-9]{8,}\b', ' ', name)
    
    # Remove reference numbers containing dashes (e.g., GB25682264-000035, V72413-23041)
    name = re.sub(r'\b[A-Z0-9]*\d[A-Z0-9]*-[A-Z0-9]+\b', ' ', name)

    
    # Standardize entities
    name = re.sub(r'\b(LTD|LIMITED|INC|CORP|LLC|PLC)\b', ' ', name)

    # Remove website prefixes and suffixes
    name = re.sub(r'\b(WWW\.|\\.COM|\\.CO\\.UK)\b', ' ', name)

    # Add this inside your clean_merchant_name function
    name = re.sub(r'\bPYMT\b', 'PAYMENT', name)
    name = re.sub(r'\bS/MKTS\b', 'SUPERMARKET', name)
    name = re.sub(r'\bTXN\b', 'TRANSACTION', name)
    
    # Clean up whitespace
    name = ' '.join(name.split())
    
    return name


if len(df_tx) > 0:
    df_tx['cleaned_description'] = df_tx['description'].apply(clean_merchant_name)
    display(df_tx[['description', 'cleaned_description']].head())

,description,cleaned_description
0,PAYPAL PAYMENT,PAYMENT
1,AMZNMktplace*CV3L1 \tON 03 DEC BC,AMZNMKTPLACE ON 03 DEC BC
2,CEX \tON 01 JUN CLP,CEX ON 01 JUN CLP
3,SPEEDWELL PHARMACY \tON 16 JUN CL,SPEEDWELL PHARMACY ON 16 JUN CL
4,SAINSBURYS S/MKTS \tON 03 NOV CLP,SAINSBURYS SUPERMARKET ON 03 NOV CLP


In [59]:
def remove_dd_mmm_dates(name):
    if not isinstance(name, str) or not name: return ""
    
    # 1. Remove dd MMM dates (e.g., ON 03 DEC, 01 JUN)
    name = re.sub(r'\b(?:ON\s+)?\d{1,2}\s(?:JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)\b', '', name)
    
    # 2. Remove common UK bank transaction codes (usually standalone words)
    # BCC=Bank Credit Card, CLP/CPM=Contactless, DDR=Direct Debit, FT=Funds Transfer, etc.
    name = re.sub(r'\b(?:BCC|CLP|CPM|DDR|DD|FT|BC|CL|CP|DC|SST|UNP|ASD)\b', '', name)
    
    # 3. Clean up any excess whitespace left behind
    name = ' '.join(name.split())
    return name


if len(df_tx) > 0:
    df_tx['cleaned_description'] = df_tx['cleaned_description'].apply(remove_dd_mmm_dates)
    display(df_tx[['description', 'cleaned_description']].head())


,description,cleaned_description
0,PAYPAL PAYMENT,PAYMENT
1,AMZNMktplace*CV3L1 \tON 03 DEC BC,AMZNMKTPLACE
2,CEX \tON 01 JUN CLP,CEX
3,SPEEDWELL PHARMACY \tON 16 JUN CL,SPEEDWELL PHARMACY
4,SAINSBURYS S/MKTS \tON 03 NOV CLP,SAINSBURYS SUPERMARKET


In [60]:
# Export all transactions with their matched labels to a CSV for easy bulk viewing
df_tx[['description', 'cleaned_description']].to_csv('cleaned_transactions_preview.csv', index=False)
print(f"Saved {len(df_tx)} rows to cleaned_transactions_preview.csv!")


Saved 3300 rows to cleaned_transactions_preview.csv!


## 2. Feature Engineering: Semantic Embeddings

Use lightweight NLP model to catch semantic relationships instead of just TF-IDF.

In [61]:
# Load lightweight embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for cleaned descriptions
if len(df_tx) > 0:
    print("Generating embeddings (this may take a moment)...")
    embeddings = model.encode(df_tx['cleaned_description'].tolist())
    print(f"Generated embeddings of shape {embeddings.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6958.53it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings (this may take a moment)...
Generated embeddings of shape (3300, 384)


In [62]:
# Create DBSCAN instance
# eps defines the maximum distance between two samples to be grouped
# min_samples is the minimum number of transactions to form a valid merchant cluster
clustering_model = DBSCAN(eps=0.3, min_samples=3, metric='cosine')

# Fit the model to your generated embeddings
cluster_labels = clustering_model.fit_predict(embeddings)

# Assign the predicted categories back to your DataFrame
df_tx['merchant_cluster_id'] = cluster_labels

# View the results (excluding noise points labeled as -1 which didn't fit into any cluster)
clustered_tx = df_tx[df_tx['merchant_cluster_id'] != -1]

# Display the clusters so you can verify if the categoriser worked!
display(clustered_tx[['description', 'cleaned_description', 'merchant_cluster_id']].sort_values('merchant_cluster_id').head(30))


,description,cleaned_description,merchant_cluster_id
0,PAYPAL PAYMENT,PAYMENT,0
35,PAYPAL PAYMENT,PAYMENT,0
87,"Payment, Thank You","PAYMENT, THANK YOU",0
1758,PAYMENT RECEIVED,PAYMENT RECEIVED,0
93,"Payment, Thank You","PAYMENT, THANK YOU",0
1177,PAYMENT RECEIVED - THANK YOU,PAYMENT RECEIVED - THANK YOU,0
2047,PAYMENT RECEIVED,PAYMENT RECEIVED,0
1194,PAYMENT RECEIVED - THAN,PAYMENT RECEIVED - THAN,0
2061,PAYPAL PAYMENT,PAYMENT,0
3107,PAYMENT RECEIVED,PAYMENT RECEIVED,0


In [63]:
# Export all transactions with their matched labels to a CSV for easy bulk viewing
df_tx[['description', 'cleaned_description', 'merchant_cluster_id']].sort_values('merchant_cluster_id').to_csv('cleaned_transactions_preview.csv', index=False)
print(f"Saved {len(df_tx)} rows to cleaned_transactions_preview.csv!")

Saved 3300 rows to cleaned_transactions_preview.csv!


## 3. Categorization / Classification

We can use the  embeddings we already generated to map either clusters or individual transactions to pre-defined spending categories.

In [64]:
import re
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Mapping of short names to ML descriptions (for the ML fallback)
category_map = {
    "Housing": "Housing rent and mortgage payments",
    "Utilities": "Household Utilities electricity water and gas",
    "Telecom": "Telecom internet broadband and mobile phone bills",
    "Subscriptions": "Recurring subscriptions software gym and streaming",
    "Insurance": "Insurance premiums auto home and health",
    "Groceries": "Groceries and supermarket shopping",
    "Dining": "Restaurants dining out and coffee shops",
    "Delivery": "Food delivery and takeaways",
    "Transport": "Public transport trains and buses",
    "Auto": "Auto expenses fuel parking and maintenance",
    "Shopping": "Retail shopping clothing electronics and home",
    "Entertainment": "Entertainment events and cinema",
    "Health": "Health pharmacy and personal care",
    "Travel": "Travel flights and hotels",
    "Bank Fees": "Bank fees and financial charges",
    "Transfers": "Money transfers and savings",
    "Income": "Income salary and refunds",
    "Household": "Repairs, DIY, decorating, pet care",
    "Unknown": "Cash widthdrawls"
}
category_labels = list(category_map.keys())
category_descriptions = list(category_map.values())

# 2. Deterministic Rules Engine
# Standard \b indicates word-boundaries to prevent partial matches
regex_rules = {
    r'\b(?:TESCO|SAINSBURYS|ASDA|WAITROSE|ALDI|LIDL)\b': 'Groceries',
    r'\b(?:HSBC|BARCLAYS|LLOYDS|NATWEST|HALIFAX|SANTANDER|MONZO|MOORCROFT GROUP)\b': 'Bank Fees',
    r'\b(?:MCDONALDS|KFC|BURGER KING|NANDOS|COSTA|STARBUCKS|UPPERCRUST|FULLER SMITH|JD WETHERSPOON)\b': 'Dining',
    r'\b(?:UBER|UBEREATS|DELIVEROO|JUST EAT)\b': 'Delivery',
    r'\b(?:TFL|TRAINLINE|RAIL|ARRIVA|STAGECOACH)\b': 'Transport',
    r'\b(?:NETFLIX|SPOTIFY|AMAZON PRIME|DISNEY PLUS|CURSOR, AI|UNISON|NOW)\b': 'Subscriptions',
    r'\b(?:MAX SPIELMANN|NYX|EDUBOX|EDE AND RAVENSCROF|WH SMITH|TGTG)|AMIGO @ HAMMERSMIT|MR SIMMS OLDE SWEE|WELCOME BREAK-LOND|DUNELM|SIMMONS|AMZNMKTPLACE\b': 'Shopping',
    r'\b(?:ANIMAL FRIENDS INS|COMPANION C|PETSATHOME|SWAROVSKI|PL2|)\b': 'Household',
    r'\b(DVLA-[A-Z]{2}\d{2}\s?[A-Z]{3}|SHELL)\b': 'Auto',
    r'\b(AO-OPTICALSERVICES|ADARO. VISIONEXPRE)\b': 'Health',
    r'\b(MIMEO, SA_SUPPORT)\b': 'Subscriptions',
    r'\b(SCREWFIX)\b': 'Household',
    r'\b(EE D)\b': 'Telecom',
    r'\b(NOTEMACHINE NOTEMACHINE)\b': 'Unknown'
}

# Prep output columns
df_tx['suggested_category'] = None
df_tx['category_confidence'] = 0.0
df_tx['match_type'] = None # To easily see if it was 'Rule' or 'ML'

unmatched_clusters = []
unmatched_noise_indices = []

# ==========================================
# PHASE 1: RULES ENGINE (Evaluate by Cluster)
# ==========================================
for cluster_id in df_tx['merchant_cluster_id'].unique():
    bool_idx = df_tx['merchant_cluster_id'] == cluster_id
    cluster_texts = df_tx.loc[bool_idx, 'cleaned_description'].tolist()
    
    # Handle the noise transactions (-1) individually
    if cluster_id == -1:
        for idx in df_tx[bool_idx].index:
            text = df_tx.loc[idx, 'cleaned_description']
            matched = False
            for pattern, category in regex_rules.items():
                if re.search(pattern, text):
                    df_tx.at[idx, 'suggested_category'] = category
                    df_tx.at[idx, 'category_confidence'] = 1.0
                    df_tx.at[idx, 'match_type'] = 'Rule'
                    matched = True
                    break
            if not matched:
                unmatched_noise_indices.append(idx)
        continue
    
    # Handle valid, cohesive clusters
    # If ANY transaction in the cluster matches a rule, assign the whole cluster!
    cluster_resolved = False
    for pattern, category in regex_rules.items():
        if any(re.search(pattern, text) for text in cluster_texts):
            df_tx.loc[bool_idx, 'suggested_category'] = category
            df_tx.loc[bool_idx, 'category_confidence'] = 1.0
            df_tx.loc[bool_idx, 'match_type'] = 'Rule'
            cluster_resolved = True
            break # Stop checking rules once it matches
            
    if not cluster_resolved:
        unmatched_clusters.append(cluster_id)

print(f"Phase 1: Rules Engine safely matched {len(df_tx[df_tx['match_type'] == 'Rule'])} transactions.")

# ==========================================
# PHASE 2: ML EMBEDDINGS (For leftovers only)
# ==========================================
# Embed category dictionary 
category_embeddings = model.encode(category_descriptions)

# Evaluate unmatched noise points individually
if unmatched_noise_indices:
    noise_embeddings = embeddings[unmatched_noise_indices]
    sim_matrix = cosine_similarity(noise_embeddings, category_embeddings)
    best_indices = np.argmax(sim_matrix, axis=1)
    best_scores = np.round(np.max(sim_matrix, axis=1), 2)
    
    df_tx.loc[unmatched_noise_indices, 'suggested_category'] = [category_labels[i] for i in best_indices]
    df_tx.loc[unmatched_noise_indices, 'category_confidence'] = best_scores
    df_tx.loc[unmatched_noise_indices, 'match_type'] = 'ML'

# Evaluate unmatched cohesive clusters via Centroid
for cluster_id in unmatched_clusters:
    bool_idx = df_tx['merchant_cluster_id'] == cluster_id
    cluster_embeddings = embeddings[bool_idx]
    
    # Calculate Centroid
    centroid = np.mean(cluster_embeddings, axis=0).reshape(1, -1)
    
    # Cosine Similarity
    sim_matrix = cosine_similarity(centroid, category_embeddings)
    best_idx = np.argmax(sim_matrix, axis=1)[0]
    best_score = np.round(np.max(sim_matrix, axis=1)[0], 2)
    
    # Bulk assign the single result to ALL transactions in this cluster
    df_tx.loc[bool_idx, 'suggested_category'] = category_labels[best_idx]
    df_tx.loc[bool_idx, 'category_confidence'] = best_score
    df_tx.loc[bool_idx, 'match_type'] = 'ML'

print(f"Phase 2: Semantic ML matched the remaining {len(df_tx[df_tx['match_type'] == 'ML'])} transactions.")

# Review results
display_cols = ['cleaned_description', 'merchant_cluster_id', 'match_type', 'suggested_category', 'category_confidence']
display(df_tx[display_cols].sort_values(['match_type', 'merchant_cluster_id']).sample(10))

Phase 1: Rules Engine safely matched 1321 transactions.
Phase 2: Semantic ML matched the remaining 1979 transactions.


,cleaned_description,merchant_cluster_id,match_type,suggested_category,category_confidence
3109,PHOENIX,56,ML,Groceries,0.21
503,PAYBYPHONE-2Z3TDOP,129,ML,Telecom,0.44
73,INTEREST CHARGED,34,ML,Bank Fees,0.46
632,INTEREST CHARGED,34,ML,Bank Fees,0.46
329,D&AMP;G APPLIANCE CARE,53,ML,Household,0.34
993,NON-STERLING TRANS FEE,17,ML,Bank Fees,0.42
3093,PAYMENT,0,ML,Transfers,0.37
756,TESCO STORES,14,Rule,Groceries,1.00
2094,CUCURUCHO AZCA SPAIN AMOUNT IN E,179,ML,Unknown,0.24
2154,SAMANTHA RUSSELL DOLLAR- NOV/DE,75,ML,Unknown,0.27


In [65]:
# Export all transactions with their suggested categories to CSV
output_cols = [
    'description', 
    'cleaned_description', 
    'merchant_cluster_id', 
    'suggested_category', 
    'category_confidence'
]
# Round the confidence score to 2 decimal places
df_tx['category_confidence'] = df_tx['category_confidence'].round(3)
# Sort by the suggested category then the cluster id logically group the csv layout
df_tx[output_cols].sort_values(['category_confidence', 'suggested_category', 'merchant_cluster_id']).to_csv('cleaned_transactions_preview.csv', index=False)
print(f"Saved {len(df_tx)} categorized rows to cleaned_transactions_preview.csv!")
low_confidence_count = len(df_tx[df_tx["category_confidence"] < 0.35])
print(f"Transactions with confidence < 0.35: {low_confidence_count}")

Saved 3300 categorized rows to cleaned_transactions_preview.csv!
Transactions with confidence < 0.35: 1397
